# ExamScheduling — Colab Reproduction Runner

Runs the full benchmark batch for CSC 2400: **13 algorithms × 8 ITC 2007 instances × 3 seeds**. Expect ~60–120 minutes on a free-tier Colab CPU runtime; a Pro runtime halves that.

**What this notebook does:**
1. Installs build deps (g++, OR-Tools, pandas, matplotlib, pytest)
2. Clones the repo + builds the headers-only C++ solver
3. Smoke-tests one algorithm × one instance to catch env breakage early
4. Loops every algorithm × every ITC 2007 set × `NUM_SEEDS` seeds, saving per-run JSON into `results/colab_batch/`
5. Aggregates results into a single CSV + regenerates the paper figures
6. Zips `results/` and triggers a browser download

> **If a step fails mid-batch:** the outer loop `continue`s past failures and still writes whatever completed. Check `results/colab_batch/failures.log` afterwards.

## 1. Environment setup

In [ ]:
!apt-get install -y build-essential > /dev/null 2>&1
!pip install -q ortools pulp numpy pandas matplotlib plotly pytest

## 1b. (optional) Mount Google Drive

Run this cell **before** the zip/download cells to auto-backup the result zips to your Drive. Skip it if you only want the browser download.

> `files.download(...)` by itself sends the zip to your computer's Downloads folder — it doesn't touch Drive. Mounting here gives you both: the Drive copy survives runtime disconnects, and the browser download is still triggered.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_DIR = '/content/drive/MyDrive/exam_scheduling_batches'
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    os.environ['DRIVE_BACKUP_DIR'] = DRIVE_BACKUP_DIR
    print(f'Drive mounted. Zips will also land in: {DRIVE_BACKUP_DIR}')
except (ImportError, ModuleNotFoundError):
    print('Not running on Colab — skipping Drive mount.')

## 2. Clone the repo

Update `GH_USER` / `REPO` if you forked. The cell is idempotent — re-running after the clone just `cd`s into the checkout.

In [ ]:
import os

GH_USER = 'Dialovos'
REPO = 'exam-scheduling-algos-analyze'

if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/{GH_USER}/{REPO}.git
%cd {REPO}
!pwd && git log -1 --oneline

## 3. Build the C++ solver + sanity smoke

In [ ]:
!make
print('\n--- smoke: Tabu on set1, seed 42 ---')
!python main.py --dataset instances/exam_comp_set1.exam --algo tabu --seed 42 --no-batch --quiet

## 4. Full batch

Drops results into `results/colab_batch/<algo>_<set>_<seed>/`. Adjust `ALGOS` / `DATASETS` / `NUM_SEEDS` below to trim scope — the default is the full paper matrix.

CP-SAT (`cpsat`) has a 60 s time limit per run; the heuristics finish in seconds. IP (`ip`) is skipped for `n > 500` exams automatically inside `main.py`.

In [ ]:
import glob, os, subprocess, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

ALGOS = ['greedy', 'tabu', 'kempe', 'sa', 'alns', 'gd', 'abc', 'ga',
         'lahc', 'woa', 'hho', 'cpsat', 'vns']
DATASETS = sorted(glob.glob('instances/exam_comp_set*.exam'))
NUM_SEEDS = 3

# Worker count: every heuristic is single-threaded C++, so pack runs across cores.
# Leave one vCPU free for the Python driver + I/O; CP-SAT inside the cpsat algo
# is internally single-worker here (num_workers=8 is only set by the Python IP path).
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)

BATCH_ROOT = Path('results/colab_batch')
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
FAIL_LOG = BATCH_ROOT / 'failures.log'
if FAIL_LOG.exists(): FAIL_LOG.unlink()


def run_one(algo, dataset, seed):
    """Child-process entry: run one (algo, dataset, seed) combination."""
    ds_name = Path(dataset).stem
    run_dir = BATCH_ROOT / f'{algo}_{ds_name}_seed{seed}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', dataset, '--algo', algo,
           '--seed', str(seed), '--no-batch', '--quiet',
           '--output', str(run_dir)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            return (algo, ds_name, seed, 'fail', r.stderr[-500:])
        return (algo, ds_name, seed, 'ok', '')
    except subprocess.TimeoutExpired:
        return (algo, ds_name, seed, 'timeout', '600s')


jobs = [(a, d, s) for a in ALGOS for d in DATASETS
        for s in range(42, 42 + NUM_SEEDS)]
total = len(jobs)
print(f'Launching {total} runs across {MAX_WORKERS} workers '
      f'({len(ALGOS)} algos × {len(DATASETS)} sets × {NUM_SEEDS} seeds)')
print(f'Output: {BATCH_ROOT.resolve()}')

started = time.time()
completed = 0
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(run_one, *j): j for j in jobs}
    for fut in as_completed(futures):
        algo, ds_name, seed, status, msg = fut.result()
        completed += 1
        elapsed = time.time() - started
        eta = elapsed / completed * (total - completed)
        tag = '  ok' if status == 'ok' else f' {status.upper()}'
        print(f'  [{completed:>3}/{total}] {algo:<7} {ds_name:<18} seed={seed}'
              f'  elapsed={elapsed/60:5.1f}m  eta={eta/60:5.1f}m {tag}',
              flush=True)
        if status != 'ok':
            with open(FAIL_LOG, 'a') as f:
                f.write(f'{algo}/{ds_name}/seed{seed}: {status}\n{msg}\n\n')

print(f'\nDone in {(time.time()-started)/60:.1f} min. Failures:',
      FAIL_LOG.stat().st_size if FAIL_LOG.exists() else 0, 'bytes')

## 5. Aggregate + regenerate figures

Walks `results/colab_batch/`, assembles one long-form DataFrame, runs the full matplotlib plot suite into `results/colab_batch/figures/`.

In [ ]:
import json, pandas as pd
from pathlib import Path
from utils.plots import generate_all_plots

BATCH_ROOT = Path('results/colab_batch')
rows = []
for run_dir in sorted(BATCH_ROOT.iterdir()):
    if not run_dir.is_dir(): continue
    sb = run_dir / 'soft_breakdown.json'
    if not sb.exists(): continue
    algo, ds_name, seed_tag = run_dir.name.split('_', 2)
    seed = int(seed_tag.removeprefix('seed'))
    breakdown = json.loads(sb.read_text())
    for algo_label, parts in breakdown.items():
        row = {'algorithm': algo_label, 'dataset': ds_name, 'seed': seed,
               'soft_penalty': sum(parts.values())}
        row.update(parts)
        rows.append(row)

df = pd.DataFrame(rows)
print(f'Aggregated {len(df)} rows')
df.to_csv(BATCH_ROOT / 'aggregated.csv', index=False)

if not df.empty:
    # Minimum columns for plotting — synthesise runtime + feasibility from the CSV
    df['runtime'] = df.get('runtime', 0.0)
    df['feasible'] = df.get('feasible', True)
    generate_all_plots(df, output_dir=str(BATCH_ROOT / 'figures'))

## 6. Zip + download

Colab sandboxes are ephemeral — do this before the runtime disconnects.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab.zip results/colab_batch/ > /dev/null

# If Drive was mounted earlier, auto-backup the zip there too.
drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab.zip')

files.download('batch_colab.zip')

## 7. Post-export IP — CP-SAT with Tabu warm-start

> **Run this only after cell #6 finishes downloading `batch_colab.zip`.**
> If the runtime dies mid-IP, the main batch is already safe on disk + Drive.

This cell reuses the Tabu solutions from `results/colab_batch/` as a CP-SAT warm-start hint (`add_hint()`) — on feasible starting points that typically gives a 5–20× speedup versus cold CP-SAT. Runs **sequentially** per instance because CP-SAT internally parallelises across all vCPUs (`--ip-workers 0` = all cores) and outer parallelism would thrash.

- `IP_TIME` — wall-clock per instance (5 minutes is a reasonable default; raise it on A100 runtime if you want tighter bounds)
- `IP_MAX_EXAMS` — instances larger than this are skipped (default 900 covers 6 of 8 ITC sets)
- Results land in `results/colab_batch_ip/ip_<setname>/` and are zipped separately as `batch_colab_ip.zip`.

In [ ]:
import glob, subprocess, time
from pathlib import Path

IP_TIME = 300          # seconds per instance
IP_MAX_EXAMS = 900     # skip instances larger than this

main_batch = Path('results/colab_batch')
ip_batch = Path('results/colab_batch_ip')
ip_batch.mkdir(parents=True, exist_ok=True)
ip_fail = ip_batch / 'failures.log'
if ip_fail.exists(): ip_fail.unlink()

datasets = sorted(glob.glob('instances/exam_comp_set*.exam'))
t0 = time.time()

for ds in datasets:
    ds_name = Path(ds).stem
    # Pick the first available Tabu solution as warm-start (any seed works)
    warm_candidates = sorted(main_batch.glob(
        f'tabu_{ds_name}_seed*/solutions/solution_tabu_search_*.sln'))
    warm = str(warm_candidates[0]) if warm_candidates else None

    run_dir = ip_batch / f'ip_{ds_name}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', ds, '--algo', 'ip',
           '--ip-time', str(IP_TIME),
           '--ip-max-exams', str(IP_MAX_EXAMS),
           '--ip-workers', '0',                 # 0 = all cores
           '--seed', '42', '--no-batch', '--quiet',
           '--output', str(run_dir)]
    if warm:
        cmd += ['--ip-warmstart', warm]

    print(f'\n=== IP on {ds_name} '
          f'(warm-start: {"yes" if warm else "no"}) ===', flush=True)
    try:
        subprocess.run(cmd, timeout=IP_TIME + 180, check=False)
    except subprocess.TimeoutExpired:
        with open(ip_fail, 'a') as f:
            f.write(f'{ds_name}: TIMEOUT {IP_TIME + 180}s\n\n')
        print(f'  TIMEOUT', flush=True)
    print(f'  total elapsed: {(time.time()-t0)/60:.1f}m', flush=True)

print(f'\nIP batch complete in {(time.time()-t0)/60:.1f}m.')

## 8. Zip + download IP results

Separate zip so you can choose to download just this piece. If Drive is mounted, it also lands there automatically.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab_ip.zip results/colab_batch_ip/ > /dev/null

drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab_ip.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab_ip.zip')

files.download('batch_colab_ip.zip')